# Semantic Model Similarity

Find possible duplicates and schema subsets using three symmetric scores: **schema similarity** (the existing six structural/text signals), **security similarity** (role rules and RLS propagation), and **combined similarity** (95% schema + 5% security by default). Roles match one-to-one by rules, independently of their names. Directional **schema containment** remains separate and excludes security.

When both complete scans find no roles, security is not applicable and combined equals schema. Missing or incomplete security makes security and combined unavailable. Duplicate labels and groups follow combined-score thresholds with warnings for differing security; differences do not veto grouping.

**Needs:** the catalog tables from **001_semantic_model_tom_catalog** in the same attached Lakehouse, including `semantic_model_security` for security scoring. The Fabric runtime supplies pandas, scikit-learn, and SciPy. All computation is local to the notebook, with no external embedding endpoint. Open **003_semantic_model_similarity_results** after scoring.

## Parameters

Tier thresholds and signal weights. The cell below is collapsed to keep the results close — click **Show input** to adjust it, then re-run.

In [ ]:
# Results are read from and written to the lakehouse attached to this notebook.

# Blocking limits comparisons to model pairs that share at least one table or measure name.
# Disable to force full pairwise comparison (slower on large catalogs).
ENABLE_BLOCKING = True

# Combined-score tier thresholds.
DUPLICATE_THRESHOLD = 0.95
SIMILAR_THRESHOLD = 0.70

# Containment threshold. Containment is a second, directional score that answers a
# different question than similarity: "does one model contain the schema definitions
# of the other?" Security is not part of directional containment.
CONTAINMENT_THRESHOLD = 0.95

# Report knobs.
TOP_N = 20  # Rows shown in the ranked pair table.
HEATMAP_MIN_SCORE = SIMILAR_THRESHOLD  # Hide heatmap cells scoring below this combined value.

# Relative weights for each schema similarity signal. Values are normalized.
SIMILARITY_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_dax_embedding": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

COMBINED_WEIGHTS = {"schema": 0.95, "security": 0.05}
SECURITY_WEIGHTS = {"role_definitions": 4.0, "rls_propagation": 1.0}
SECURITY_ROLE_WEIGHTS = {"model_permission": 1.0, "rls": 1.0, "table_ols": 1.0, "column_ols": 1.0}
SCORE_VERSION = 2

# Relative weights for each containment signal. Containment uses exact measure
# definitions (name + comment-stripped DAX) instead of the TF-IDF embedding, because
# lexical cosine similarity is symmetric and cannot establish that one model's measures
# are a subset of another's. Weights are normalized per direction over whichever signals
# the source model actually has.
CONTAINMENT_WEIGHTS = {
    "tables": 0.15,
    "columns": 0.20,
    "measure_names": 0.15,
    "measure_definitions": 0.25,
    "relationships": 0.15,
    "datasources": 0.10,
}

# Guard against misconfigured weights before any scoring runs.
for _weights_name, _weights in (
    ("SIMILARITY_WEIGHTS", SIMILARITY_WEIGHTS),
    ("CONTAINMENT_WEIGHTS", CONTAINMENT_WEIGHTS),
    ("COMBINED_WEIGHTS", COMBINED_WEIGHTS),
    ("SECURITY_WEIGHTS", SECURITY_WEIGHTS),
    ("SECURITY_ROLE_WEIGHTS", SECURITY_ROLE_WEIGHTS),
):
    if any(not isinstance(weight, (int, float)) or isinstance(weight, bool) or not 0 <= weight < float("inf") for weight in _weights.values()):
        raise ValueError(f"{_weights_name} must contain finite nonnegative weights.")
    if sum(_weights.values()) <= 0:
        raise ValueError(f"{_weights_name} must sum to a positive value.")


## Prepare the data

Loads the catalog and scores every model pair. All inputs here are hidden — expand any cell to inspect the logic; you don't need to change anything.

In [ ]:
import itertools
import re
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
import hashlib
import json
import math


def security_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True, allow_nan=False)


def normalize_score_weights(weights):
    if not weights or any(isinstance(value, bool) or not math.isfinite(value) or value < 0 for value in weights.values()):
        raise ValueError("Score weights must be finite nonnegative numbers.")
    total = math.fsum(weights.values())
    if total <= 0:
        raise ValueError("Score weights must have a positive sum.")
    return {key: value / total for key, value in weights.items()}


def build_security_signature(identity, snapshots, load_status="available"):
    result = {"status": load_status, "fingerprint": None, "roles": [], "propagation": [], "definition_json": None}
    if load_status != "available":
        return result
    if len(snapshots) != 1:
        result["status"] = "not_collected" if not snapshots else "inconsistent"
        return result
    snapshot = snapshots[0]
    if (not isinstance(identity.get("catalog_scan_id"), str) or not identity["catalog_scan_id"]
            or any(snapshot.get(key) != identity.get(key) for key in ("model_id", "workspace_id", "catalog_scan_id"))):
        result["status"] = "inconsistent"
        return result
    if snapshot.get("security_schema_version") != 1:
        result["status"] = "unsupported"
        return result
    if snapshot.get("scan_status") != "complete":
        result["status"] = "incomplete"
        return result

    def text(value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError("Missing security identity")
        return value

    def permission(value, allowed):
        if value not in allowed:
            raise ValueError("Invalid security permission")
        return value

    try:
        definition = json.loads(snapshot["definition_json"])
        if not isinstance(definition["roles"], list) or not isinstance(definition["relationships"], list):
            raise ValueError("Invalid security inventory")
        roles, role_names = [], set()
        filter_count = table_ols_count = column_ols_count = 0
        for role in definition["roles"]:
            role_name = text(role["name"])
            if role_name in role_names or not isinstance(role["tables"], list):
                raise ValueError("Invalid role inventory")
            role_names.add(role_name)
            model_permission = permission(role["model_permission"], {"None", "Read", "ReadRefresh", "Refresh", "Administrator"})
            rls, table_ols, column_ols, table_names = set(), set(), set(), set()
            for table in role["tables"]:
                table_name = text(table["table"])
                if table_name in table_names or not isinstance(table["columns"], list):
                    raise ValueError("Invalid table permission inventory")
                table_names.add(table_name)
                expression = table["filter_expression"]
                if not isinstance(expression, str):
                    raise ValueError("Invalid RLS definition")
                if expression.strip():
                    rls.add((table_name, expression))
                    filter_count += 1
                table_permission = permission(table["metadata_permission"], {"Default", "None", "Read"})
                if table_permission != "Default":
                    table_ols.add((table_name, table_permission))
                table_ols_count += table_permission == "None"
                column_names = set()
                for column in table["columns"]:
                    column_name = text(column["column"])
                    if column_name in column_names:
                        raise ValueError("Duplicate column permission")
                    column_names.add(column_name)
                    column_permission = permission(column["metadata_permission"], {"Default", "None", "Read"})
                    if column_permission != "Default":
                        column_ols.add((table_name, column_name, column_permission))
                    column_ols_count += column_permission == "None"
            bundle = {
                "model_permission": model_permission,
                "rls": sorted(rls), "table_ols": sorted(table_ols), "column_ols": sorted(column_ols),
            }
            role_key = security_json(bundle)
            roles.append({"name": role_name, "key": role_key, **bundle})
        roles.sort(key=lambda role: (role["key"], role["name"]))
        propagation = set()
        for relationship in definition["relationships"]:
            endpoints = tuple(text(relationship[key]) for key in ("from_table", "from_column", "to_table", "to_column"))
            if type(relationship["is_active"]) is not bool:
                raise ValueError("Invalid relationship active state")
            settings = (
                relationship["is_active"],
                permission(relationship["cross_filtering_behavior"], {"OneDirection", "BothDirections", "Automatic"}),
                permission(relationship["security_filtering_behavior"], {"OneDirection", "BothDirections", "None"}),
                permission(relationship["from_cardinality"], {"One", "Many"}),
                permission(relationship["to_cardinality"], {"One", "Many"}),
            )
            if filter_count:
                propagation.add(endpoints + settings)
        expected_counts = {"role_count": len(roles), "rls_filter_count": filter_count, "table_ols_count": table_ols_count, "column_ols_count": column_ols_count}
        if any(snapshot.get(key) != value for key, value in expected_counts.items()):
            raise ValueError("Security inventory counts do not agree")
        canonical = security_json({"version": 1, "roles": [role["key"] for role in roles], "propagation": sorted(propagation)})
        for role in roles:
            role["fingerprint"] = hashlib.sha256(role.pop("key").encode("utf-8")).hexdigest()
        result.update({
            "status": "complete", "fingerprint": hashlib.sha256(canonical.encode("utf-8")).hexdigest(),
            "roles": roles, "propagation": sorted(propagation), **expected_counts,
            "definition_json": security_json({"roles": roles, "propagation": sorted(propagation)}),
        })
    except (KeyError, TypeError, ValueError, AttributeError):
        result["status"] = "inconsistent"
    return result


def security_overlap(left, right):
    union = left | right
    return len(left & right) / len(union) if union else None


def security_weighted_average(components, weights):
    applicable = {key: weights[key] for key, value in components.items() if value is not None}
    effective = normalize_score_weights(applicable)
    return math.fsum(components[key] * weight for key, weight in effective.items()), effective


def score_security_pair(left, right, role_weights, security_weights):
    evidence = {"components": {}, "effective_weights": {}, "role_matches": [], "unmatched_roles_a": [], "unmatched_roles_b": []}
    if left["status"] != "complete" or right["status"] != "complete":
        return {"score": None, "status": "unknown", "evidence": evidence}
    roles_a, roles_b = left["roles"], right["roles"]
    if not roles_a and not roles_b:
        return {"score": None, "status": "not_applicable", "evidence": evidence}
    comparisons = []
    for role_a in roles_a:
        row = []
        for role_b in roles_b:
            components = {"model_permission": float(role_a["model_permission"] == role_b["model_permission"])}
            for family in ("rls", "table_ols", "column_ols"):
                components[family] = security_overlap({tuple(fact) for fact in role_a[family]}, {tuple(fact) for fact in role_b[family]})
            value, effective = security_weighted_average(components, role_weights)
            row.append({"score": value, "components": components, "effective_weights": effective})
        comparisons.append(row)
    matches = []
    if roles_a and roles_b:
        from scipy.optimize import linear_sum_assignment

        indices_a, indices_b = linear_sum_assignment([[-item["score"] for item in row] for row in comparisons])
        matches = [{"role_a": int(index_a), "role_b": int(index_b), **comparisons[index_a][index_b]}
                   for index_a, index_b in zip(indices_a, indices_b)]
    matched_a = {match["role_a"] for match in matches}
    matched_b = {match["role_b"] for match in matches}
    role_score = math.fsum(match["score"] for match in matches) / max(len(roles_a), len(roles_b))
    propagation_score = security_overlap({tuple(fact) for fact in left["propagation"]}, {tuple(fact) for fact in right["propagation"]})
    components = {"role_definitions": role_score, "rls_propagation": propagation_score}
    value, effective = security_weighted_average(components, security_weights)
    evidence.update({
        "components": components, "effective_weights": effective, "role_matches": matches,
        "unmatched_roles_a": [index for index in range(len(roles_a)) if index not in matched_a],
        "unmatched_roles_b": [index for index in range(len(roles_b)) if index not in matched_b],
    })
    return {"score": value, "status": "match" if left["fingerprint"] == right["fingerprint"] else "different", "evidence": evidence}


def combine_similarity(schema_score, security_result, weights):
    effective = normalize_score_weights(weights)
    if set(effective) != {"schema", "security"}:
        raise ValueError("Combined weights must specify schema and security.")
    result = {"combined_score": None, "score_mode": "unavailable_security", "effective_weights": {}}
    if not isinstance(schema_score, (int, float)) or not math.isfinite(schema_score) or not 0 <= schema_score <= 1:
        return result
    if security_result["status"] == "not_applicable":
        result.update(combined_score=schema_score, score_mode="schema_only_no_security", effective_weights={"schema": 1.0, "security": 0.0})
    elif security_result["status"] in ("match", "different"):
        security_score = security_result["score"]
        if isinstance(security_score, (int, float)) and math.isfinite(security_score) and 0 <= security_score <= 1:
            combined = math.fsum((effective["schema"] * schema_score, effective["security"] * security_score))
            result.update(combined_score=combined, score_mode="combined", effective_weights=effective)
    return result


def classify_similarity(value, duplicate_threshold, similar_threshold):
    if value is None or not math.isfinite(value) or not 0 <= value <= 1:
        return "unassessed"
    if value >= duplicate_threshold:
        return "duplicate"
    if value >= similar_threshold:
        return "similar"
    return "distinct"


In [ ]:
def load_delta(table_name):
    return spark.table(table_name).toPandas()


models_df = load_delta("semantic_models")
tables_df = load_delta("semantic_model_tables")
columns_df = load_delta("semantic_model_columns")
relationships_df = load_delta("semantic_model_relationships")
measures_df = load_delta("semantic_model_measures")
datasources_df = load_delta("semantic_model_datasources")

if models_df.empty:
    raise ValueError(
        "No rows in the semantic_models table of the attached lakehouse. "
        "Run the TOM catalog notebook first."
    )

print(f"Models: {len(models_df)}")
print(
    f"Tables: {len(tables_df)} | Columns: {len(columns_df)} | "
    f"Measures: {len(measures_df)} | Relationships: {len(relationships_df)} | "
    f"Datasources: {len(datasources_df)}"
)

In [ ]:
def norm(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().casefold()


def norm_dax(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"/\*.*?\*/", " ", text, flags=re.S)  # block comments
    text = re.sub(r"//.*", " ", text)  # line comments
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def model_label(sig):
    return f"{sig['workspace_name']} / {sig['model_name']}"


signatures = {}
for _, row in models_df.iterrows():
    model_id = str(row["model_id"])
    signatures[model_id] = {
        "model_id": model_id,
        "workspace_id": str(row.get("workspace_id", "")),
        "workspace_name": str(row.get("workspace_name", "")),
        "model_name": str(row.get("model_name", "")),
        "tables": set(),
        "columns": set(),
        "measure_names": set(),
        "measure_definitions": set(),
        "relationships": set(),
        "datasources": set(),
        "dax_docs": [],
    }


def sig_for(model_id):
    return signatures.get(str(model_id))


for _, row in tables_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["tables"].add(norm(row["table_name"]))

for _, row in columns_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        sig["columns"].add(f"{norm(row['table_name'])}.{norm(row['column_name'])}")

for _, row in measures_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        measure_name = norm(row["measure_name"])
        measure_dax = norm_dax(row.get("expression"))
        sig["measure_names"].add(measure_name)
        # Name + DAX key so containment only credits measures whose logic also matches.
        sig["measure_definitions"].add(f"{measure_name} :: {measure_dax}")
        sig["dax_docs"].append(f"{measure_name} {measure_dax}".strip())

for _, row in relationships_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        key = (
            f"{norm(row['from_table'])}.{norm(row['from_column'])}"
            f"->{norm(row['to_table'])}.{norm(row['to_column'])}"
        )
        sig["relationships"].add(key)

for _, row in datasources_df.iterrows():
    sig = sig_for(row["model_id"])
    if sig is not None:
        conn = row.get("connection_string") or row.get("connection_details") or row.get("datasource_name")
        conn_norm = norm(conn)
        if conn_norm:
            sig["datasources"].add(conn_norm)

# Build the embedding document per model, with a structural fallback when no measures exist.
for sig in signatures.values():
    parts = list(sig["dax_docs"])
    if not parts:
        parts = sorted(sig["tables"]) + sorted(sig["columns"])
    sig["doc"] = " \n ".join(parts) if parts else (sig["model_name"] or sig["model_id"])

model_ids = list(signatures.keys())
print(f"Built signatures for {len(model_ids)} models.")


In [ ]:
def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 0.0
    union = len(set_a | set_b)
    return len(set_a & set_b) / union if union else 0.0


def coverage(source_set, other_set):
    # Directional: the fraction of source_set's members that also appear in other_set.
    # Returns None when the signal is absent from the source, so it can be excluded from
    # the weighted average rather than counted as a spurious perfect match.
    if not source_set:
        return None
    return len(source_set & other_set) / len(source_set)


def weighted_containment(source_sig, other_sig, weights):
    # How completely source_sig is contained in other_sig: a weighted mean of the
    # per-signal coverages, normalized over whichever signals the source actually has.
    accumulated = 0.0
    total_weight = 0.0
    for signal, weight in weights.items():
        signal_coverage = coverage(source_sig[signal], other_sig[signal])
        if signal_coverage is None:
            continue
        accumulated += weight * signal_coverage
        total_weight += weight
    return accumulated / total_weight if total_weight else 0.0


def classify_containment(a_in_b, b_in_a, threshold):
    a_contained = a_in_b >= threshold
    b_contained = b_in_a >= threshold
    if a_contained and b_contained:
        return "equivalent"
    if b_contained:
        return "model_a_contains_model_b"
    if a_contained:
        return "model_b_contains_model_a"
    return "partial_overlap"


if ENABLE_BLOCKING:
    block_index = defaultdict(set)
    for model_id, sig in signatures.items():
        for table_name in sig["tables"]:
            block_index[("t", table_name)].add(model_id)
        for measure_name in sig["measure_names"]:
            block_index[("m", measure_name)].add(model_id)
    candidate_pairs = set()
    for group in block_index.values():
        if len(group) > 1:
            candidate_pairs.update(itertools.combinations(sorted(group), 2))
else:
    candidate_pairs = set(itertools.combinations(sorted(model_ids), 2))

print(f"Candidate pairs to score: {len(candidate_pairs)}")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF avoids the torch/transformers dependency chain (the Fabric runtime ships
# an older PyTorch than recent transformers require). Rows are L2-normalized, so
# cosine similarity stays a plain dot product for the downstream scoring step.
docs = [signatures[model_id]["doc"] for model_id in model_ids]
vectorizer = TfidfVectorizer(min_df=1, norm="l2")

embeddings = vectorizer.fit_transform(docs).toarray()
print(f"Encoded {len(docs)} model documents into {embeddings.shape[1]}-dim TF-IDF vectors.")
embedding_index = {model_id: idx for idx, model_id in enumerate(model_ids)}

In [ ]:
from uuid import uuid4
from pyspark.sql.utils import AnalysisException

analysis_run_id = str(uuid4())
security_rows, security_load_status = [], "available"
try:
    security_rows = load_delta("semantic_model_security").to_dict("records")
except AnalysisException as error:
    error_class = getattr(error, "getErrorClass", lambda: "")()
    security_load_status = "not_collected" if error_class in ("TABLE_OR_VIEW_NOT_FOUND", "PATH_NOT_FOUND", "DELTA_PATH_DOES_NOT_EXIST") else "unavailable"

security_by_model = defaultdict(list)
identities_by_model = defaultdict(list)
for row in security_rows:
    security_by_model[str(row.get("model_id"))].append(row)
for row in models_df.to_dict("records"):
    identities_by_model[str(row["model_id"])].append(row)
for model_id, sig in signatures.items():
    identities = identities_by_model[model_id]
    identity = identities[0]
    scan_id = identity.get("catalog_scan_id")
    sig["catalog_scan_id"] = scan_id if len(identities) == 1 and isinstance(scan_id, str) and scan_id else None
    sig["security"] = build_security_signature(
        identity, security_by_model[model_id], security_load_status if len(identities) == 1 else "inconsistent"
    )
catalog_scan_ids = {sig["catalog_scan_id"] for sig in signatures.values()}
source_catalog_scan_id = next(iter(catalog_scan_ids)) if len(catalog_scan_ids) == 1 else None

weight_sum = sum(SIMILARITY_WEIGHTS.values())
pair_rows = []

for model_id_a, model_id_b in sorted(candidate_pairs):
    sig_a = signatures[model_id_a]
    sig_b = signatures[model_id_b]

    j_tables = jaccard(sig_a["tables"], sig_b["tables"])
    j_columns = jaccard(sig_a["columns"], sig_b["columns"])
    j_measures = jaccard(sig_a["measure_names"], sig_b["measure_names"])
    j_relationships = jaccard(sig_a["relationships"], sig_b["relationships"])
    j_datasources = jaccard(sig_a["datasources"], sig_b["datasources"])
    cosine = float(
        np.dot(embeddings[embedding_index[model_id_a]], embeddings[embedding_index[model_id_b]])
    )
    cosine = max(0.0, min(1.0, cosine))

    composite = (
        SIMILARITY_WEIGHTS["tables"] * j_tables
        + SIMILARITY_WEIGHTS["columns"] * j_columns
        + SIMILARITY_WEIGHTS["measure_names"] * j_measures
        + SIMILARITY_WEIGHTS["measure_dax_embedding"] * cosine
        + SIMILARITY_WEIGHTS["relationships"] * j_relationships
        + SIMILARITY_WEIGHTS["datasources"] * j_datasources
    ) / weight_sum

    # Directional containment: how much of each model is absorbed by the other.
    a_in_b = weighted_containment(sig_a, sig_b, CONTAINMENT_WEIGHTS)
    b_in_a = weighted_containment(sig_b, sig_a, CONTAINMENT_WEIGHTS)
    containment_score = max(a_in_b, b_in_a)
    containment_relationship = classify_containment(a_in_b, b_in_a, CONTAINMENT_THRESHOLD)

    security_result = score_security_pair(sig_a["security"], sig_b["security"], SECURITY_ROLE_WEIGHTS, SECURITY_WEIGHTS)
    combination = combine_similarity(float(composite), security_result, COMBINED_WEIGHTS)
    combined = combination["combined_score"]
    tier = classify_similarity(combined, DUPLICATE_THRESHOLD, SIMILAR_THRESHOLD)
    security_evidence = {**security_result["evidence"], "combined_effective_weights": combination["effective_weights"]}

    pair_rows.append({
        "model_id_a": model_id_a,
        "model_a": model_label(sig_a),
        "workspace_a": sig_a["workspace_name"],
        "model_id_b": model_id_b,
        "model_b": model_label(sig_b),
        "workspace_b": sig_b["workspace_name"],
        "same_model_name": norm(sig_a["model_name"]) == norm(sig_b["model_name"]),
        "cross_workspace": sig_a["workspace_id"] != sig_b["workspace_id"],
        "jaccard_tables": round(j_tables, 4),
        "jaccard_columns": round(j_columns, 4),
        "jaccard_measure_names": round(j_measures, 4),
        "jaccard_relationships": round(j_relationships, 4),
        "jaccard_datasources": round(j_datasources, 4),
        "dax_embedding_cosine": round(cosine, 4),
        "composite_score": round(composite, 4),
        "schema_score": float(composite),
        "security_score": security_result["score"],
        "combined_score": combined,
        "score_mode": combination["score_mode"],
        "security_comparison_status": security_result["status"],
        "security_evidence_json": security_json(security_evidence),
        "security_fingerprint_a": sig_a["security"]["fingerprint"],
        "security_fingerprint_b": sig_b["security"]["fingerprint"],
        "catalog_scan_id_a": sig_a["catalog_scan_id"],
        "catalog_scan_id_b": sig_b["catalog_scan_id"],
        "analysis_run_id": analysis_run_id,
        "score_version": SCORE_VERSION,
        "containment_score": round(containment_score, 4),
        "containment_relationship": containment_relationship,
        "model_a_in_model_b": round(a_in_b, 4),
        "model_b_in_model_a": round(b_in_a, 4),
        "tier": tier,
    })

pairs_df = pd.DataFrame(pair_rows)
if not pairs_df.empty:
    pairs_df = pairs_df.sort_values(["combined_score", "schema_score"], ascending=[False, False], na_position="last").reset_index(drop=True)

# Union-find clustering over duplicate-tier pairs.
parent = {model_id: model_id for model_id in model_ids}


def find(node):
    while parent[node] != node:
        parent[node] = parent[parent[node]]
        node = parent[node]
    return node


def union(node_a, node_b):
    root_a, root_b = find(node_a), find(node_b)
    if root_a != root_b:
        parent[root_a] = root_b


if not pairs_df.empty:
    for _, row in pairs_df[pairs_df["tier"] == "duplicate"].iterrows():
        union(row["model_id_a"], row["model_id_b"])

cluster_members = defaultdict(list)
for model_id in model_ids:
    cluster_members[find(model_id)].append(model_id)

cluster_rows = []
cluster_number = 0
for members in cluster_members.values():
    if len(members) > 1:
        cluster_number += 1
        for model_id in members:
            sig = signatures[model_id]
            cluster_rows.append({
                "cluster_id": cluster_number,
                "cluster_size": len(members),
                "model_id": model_id,
                "model": model_label(sig),
                "workspace_name": sig["workspace_name"],
                "model_name": sig["model_name"],
                "analysis_run_id": analysis_run_id,
                "score_version": SCORE_VERSION,
            })

clusters_df = pd.DataFrame(cluster_rows)

# Per-model signature summary.
signature_rows = []
for sig in signatures.values():
    security = sig["security"]
    signature_rows.append({
        "model_id": sig["model_id"],
        "workspace_id": sig["workspace_id"],
        "workspace_name": sig["workspace_name"],
        "model_name": sig["model_name"],
        "table_count": len(sig["tables"]),
        "column_count": len(sig["columns"]),
        "measure_count": len(sig["measure_names"]),
        "relationship_count": len(sig["relationships"]),
        "datasource_count": len(sig["datasources"]),
        "analysis_run_id": analysis_run_id,
        "catalog_scan_id": sig["catalog_scan_id"],
        "score_version": SCORE_VERSION,
        "security_schema_version": 1,
        "security_scan_status": security["status"],
        "security_fingerprint": security["fingerprint"],
        "security_definition_json": security["definition_json"],
        "role_count": security.get("role_count"),
        "rls_filter_count": security.get("rls_filter_count"),
        "table_ols_count": security.get("table_ols_count"),
        "column_ols_count": security.get("column_ols_count"),
    })
signatures_df = pd.DataFrame(signature_rows)

duplicate_count = int((pairs_df["tier"] == "duplicate").sum()) if not pairs_df.empty else 0
similar_count = int((pairs_df["tier"] == "similar").sum()) if not pairs_df.empty else 0
unassessed_count = int((pairs_df["tier"] == "unassessed").sum()) if not pairs_df.empty else 0
containment_count = (
    int((pairs_df["containment_score"] >= CONTAINMENT_THRESHOLD).sum()) if not pairs_df.empty else 0
)
print(f"Possible duplicate pairs: {duplicate_count} | Similar pairs: {similar_count} | Schema coverage pairs: {containment_count}")
print(f"Unassessed combined scores: {unassessed_count} | Duplicate clusters: {cluster_number}")


## Save results to the lakehouse

Write the per-model signature summary, the full pairwise scores, and the duplicate clusters as Delta tables so the analysis is queryable outside this notebook.

In [ ]:
similarity_outputs = {
    "semantic_model_signatures": signatures_df,
    "semantic_model_similarity_pairs": pairs_df,
    "semantic_model_duplicate_clusters": clusters_df,
}
similarity_schemas = {
    "semantic_model_signatures": "model_id STRING, workspace_id STRING, workspace_name STRING, model_name STRING, table_count BIGINT, column_count BIGINT, measure_count BIGINT, relationship_count BIGINT, datasource_count BIGINT, analysis_run_id STRING, catalog_scan_id STRING, score_version BIGINT, security_schema_version BIGINT, security_scan_status STRING, security_fingerprint STRING, security_definition_json STRING, role_count BIGINT, rls_filter_count BIGINT, table_ols_count BIGINT, column_ols_count BIGINT",
    "semantic_model_similarity_pairs": "model_id_a STRING, model_a STRING, workspace_a STRING, model_id_b STRING, model_b STRING, workspace_b STRING, same_model_name BOOLEAN, cross_workspace BOOLEAN, jaccard_tables DOUBLE, jaccard_columns DOUBLE, jaccard_measure_names DOUBLE, jaccard_relationships DOUBLE, jaccard_datasources DOUBLE, dax_embedding_cosine DOUBLE, composite_score DOUBLE, containment_score DOUBLE, containment_relationship STRING, model_a_in_model_b DOUBLE, model_b_in_model_a DOUBLE, tier STRING, schema_score DOUBLE, security_score DOUBLE, combined_score DOUBLE, score_mode STRING, security_comparison_status STRING, security_evidence_json STRING, security_fingerprint_a STRING, security_fingerprint_b STRING, catalog_scan_id_a STRING, catalog_scan_id_b STRING, analysis_run_id STRING, score_version BIGINT",
    "semantic_model_duplicate_clusters": "cluster_id BIGINT, cluster_size BIGINT, model_id STRING, model STRING, workspace_name STRING, model_name STRING, analysis_run_id STRING, score_version BIGINT",
}

for table_name, frame in similarity_outputs.items():
    records = frame.to_dict("records")
    integer_fields = {field.split()[0] for field in similarity_schemas[table_name].split(",") if field.split()[1] == "BIGINT"}
    for record in records:
        for field, value in record.items():
            if pd.isna(value):
                record[field] = None
            elif field in integer_fields:
                record[field] = int(value)
    spark.createDataFrame(records, schema=similarity_schemas[table_name]).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

from datetime import datetime, timezone

_run_meta = [{
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "analysis_run_id": analysis_run_id,
    "catalog_scan_id": source_catalog_scan_id,
    "score_version": SCORE_VERSION,
    "security_schema_version": 1,
    "duplicate_threshold": float(DUPLICATE_THRESHOLD),
    "similar_threshold": float(SIMILAR_THRESHOLD),
    "containment_threshold": float(CONTAINMENT_THRESHOLD),
    "enable_blocking": bool(ENABLE_BLOCKING),
    "similarity_weights_json": security_json(SIMILARITY_WEIGHTS),
    "containment_weights_json": security_json(CONTAINMENT_WEIGHTS),
    "combined_weights_json": security_json(normalize_score_weights(COMBINED_WEIGHTS)),
    "security_weights_json": security_json(SECURITY_WEIGHTS),
    "security_role_weights_json": security_json(SECURITY_ROLE_WEIGHTS),
    "model_count": len(signatures_df),
    "pair_count": len(pairs_df),
    "duplicate_count": duplicate_count,
    "similar_count": similar_count,
    "unassessed_count": unassessed_count,
    "containment_count": containment_count,
    "cluster_count": cluster_number,
}]
run_schema = "generated_at STRING, analysis_run_id STRING, catalog_scan_id STRING, score_version BIGINT, security_schema_version BIGINT, duplicate_threshold DOUBLE, similar_threshold DOUBLE, containment_threshold DOUBLE, enable_blocking BOOLEAN, similarity_weights_json STRING, containment_weights_json STRING, combined_weights_json STRING, security_weights_json STRING, security_role_weights_json STRING, model_count BIGINT, pair_count BIGINT, duplicate_count BIGINT, similar_count BIGINT, unassessed_count BIGINT, containment_count BIGINT, cluster_count BIGINT"
spark.createDataFrame(_run_meta, schema=run_schema).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("semantic_model_similarity_run")
print("Wrote versioned run metadata to semantic_model_similarity_run")


## Next steps

- Run **003_semantic_model_similarity_results** and use **Review**, **Groups**, **Compare**, and **Similarity map**. Three scores distinguish schema overlap, security-definition similarity, and the combined ranking.
- Possible duplicates follow the combined cutoff even when security differs. With the default weights, 100% schema and 0% security produce 95% combined and qualify at the inclusive 95% cutoff. Review the security warning before making consolidation decisions.
- Both models confirmed without roles use schema as combined and show security as not applicable. Missing, unreadable, partial, or inconsistent security leaves security and combined unavailable. Run **001 -> 002 -> 003** after upgrading or changing model security.
- Schema coverage remains directional and excludes security. It does not establish permission containment, matching data, or replacement safety. Matching role definitions do not prove matching assignments or effective user access.
- Adjust `SIMILARITY_WEIGHTS`, `COMBINED_WEIGHTS`, `SECURITY_WEIGHTS`, `SECURITY_ROLE_WEIGHTS`, and review thresholds in Parameters, then rerun scoring. Weights and scan provenance are recorded with the results.
- `composite_score` remains the legacy schema-only value. New consumers should use `schema_score`, `security_score`, `combined_score`, `score_mode`, and `security_comparison_status`. The versioned outputs are saved to the same attached Lakehouse.